## 1. Data Preparation

The given dataset is provided in the following format: 

```sh
Vector-cam
 ├── kenya_01
    ├── images/
    │   ├── image_13133_78265aff136f61e9dc4d3bd9ef5c98f4.jpg
    │   ├── image_13134_0439c0c62bccf0741f5a8b2f65350f56.jpg
    │   ├── image_13135_820fc7b297ad39eae9310bf8c607d97e.jpg
    │   ├── image_13136_350a6ba0669c8c14f9805c9c2c99c4da.jpg
    │   └── ...
    └── labels/
        ├── image_13133_78265aff136f61e9dc4d3bd9ef5c98f4.txt
        ├── image_13134_0439c0c62bccf0741f5a8b2f65350f56.txt
        ├── image_13135_820fc7b297ad39eae9310bf8c607d97e.txt
        ├── image_13136_350a6ba0669c8c14f9805c9c2c99c4da.txt
        └── ...
```

The recommended format differs from what we have, so we will split the dataset to support the [YOLO fromat](https://docs.ultralytics.com/datasets/detect/#ultralytics-yolo-format). For this, we will import some libraries.


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split # To split the data into train and test
import cv2
import random
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np
import glob
import torch

In [ ]:
def set_deterministic(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_deterministic(42)

In [ ]:
# Define kenya base directory
kenya_data_dir = "/kaggle/input/datasets/sumn2u/vector-cam/kenya_01"

if os.path.exists(kenya_data_dir):
    print(f"Directory found! Files in {kenya_data_dir}:")
    # List the first 5 files to check what's inside
    files = os.listdir(kenya_data_dir)
    print(files[:5]) 
else:
    print("Directory not found. Please check your dataset path.")

In [ ]:
# Define specific paths for images and labels
images_dir = os.path.join(kenya_data_dir, "images")
labels_dir = os.path.join(kenya_data_dir, "labels")

print(f"Images Dir: {images_dir}, Labels Dir: {labels_dir}")

Let's check the file extensions that we have in image directory.

In [ ]:
# Get a list of all files in the directory
files = glob.glob(os.path.join(images_dir, '*'))

# Collect unique file extensions
extensions = set()
for file in files:
    _, ext = os.path.splitext(file)
    extensions.add(ext.lower())  # Convert to lowercase to handle case sensitivity

# Print all unique extensions found
print("Unique File Extensions:")
for ext in extensions:
    print(ext)

We have all JPEG files. The below function loads images and corresponding annotation paths from directories.

In [ ]:
def load_data(images_dir, labels_dir):
    images = []
    annotations = []
    
    for filename in os.listdir(images_dir):
        if filename.endswith('.jpg'):
            image_path = os.path.join(images_dir, filename)
            annotation_path = os.path.join(labels_dir, filename.replace('.jpg', '.txt'))
            
            if os.path.exists(annotation_path):
                images.append(image_path)
                annotations.append(annotation_path)
    
    return images, annotations

In [ ]:
images, annotations = load_data(images_dir, labels_dir)

To visualizes a random sample of images. Let's create a `visualize_samples` function.

In [ ]:
def visualize_samples(images, num_samples=5):
    fig, axs = plt.subplots(1, num_samples, figsize=(15, 5))
    
    for i in range(num_samples):
        image = Image.open(images[i])
        axs[i].imshow(image)
        axs[i].axis('off')
        axs[i].set_title(f'Sample {i+1}')
    
    plt.show()

In [ ]:
visualize_samples(images)

Let's calculate and visualize the average height and width of the images in the dataset.

In [ ]:
def calculate_average_size(images_dir):
    total_width = 0
    total_height = 0
    total_images = 0

    for filename in os.listdir(images_dir):
        if filename.endswith('.jpg'):  
            image_path = os.path.join(images_dir, filename)
            image = Image.open(image_path)
            width, height = image.size
            total_width += width
            total_height += height
            total_images += 1
    
    if total_images > 0:
        average_width = total_width / total_images
        average_height = total_height / total_images
        return average_width, average_height
    else:
        return None

In [ ]:
def plot_average_size(average_width, average_height):
    categories = ['Average Width', 'Average Height']
    values = [average_width, average_height]

    plt.figure(figsize=(6, 4))
    plt.bar(categories, values, color=['skyblue', 'lightgreen'])
    plt.xlabel('Categories')
    plt.ylabel('Pixels')
    plt.title('Average Image Dimensions')
    plt.ylim(0, max(values) * 1.1)
    plt.grid(axis='y')
    plt.show()

In [ ]:
average_width, average_height = calculate_average_size(images_dir)
    
if average_width and average_height:
    print(f"Average Image Width: {average_width:.2f} pixels")
    print(f"Average Image Height: {average_height:.2f} pixels")

    # Plot average size
    plot_average_size(average_width, average_height)
else:
    print("No images found or unable to calculate average size.")

We are also going to plot the distribution of classes ('anopheles_gambiae', 'anopheles_stephensi' and 'aedes_aegypti') so, let's create a seperate function for this.

In [ ]:
def plot_class_distribution(annotations):
    classes = {'anopheles_gambiae': 0, 'anopheles_stephensi': 0, 'aedes_aegypti': 0 }
    
    for annotation_file in annotations:
        with open(annotation_file, 'r') as f:
            lines = f.readlines()
            for line in lines:
                class_label = line.split()[0]
                if class_label == '0':
                    classes['anopheles_gambiae'] += 1
                elif class_label == '1':
                    classes['anopheles_stephensi'] += 1
                elif class_label == '2':
                    classes['aedes_aegypti'] += 1
    # Print the class counts
    for cls, count in classes.items():
        print(f"{cls}: {count}")
    
    # Plotting
    plt.bar(classes.keys(), classes.values())
    plt.title('Class Distribution')
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.show()

In [ ]:
plot_class_distribution(annotations)

To draw the bounding boxes on an image based on annotations, we will be using **draw_bounding_boxes** function.

In [ ]:
def draw_bounding_boxes(image_path, annotation_path):
    image = Image.open(image_path)
    draw = ImageDraw.Draw(image)
    with open(annotation_path, 'r') as f:
        for line in f:
            class_label, x_center, y_center, width, height = map(float, line.strip().split())
            # Convert YOLO format to bounding box coordinates
            img_width, img_height = image.size
            x_center *= img_width
            y_center *= img_height
            width *= img_width
            height *= img_height
            x_min = x_center - (width / 2)
            x_max = x_center + (width / 2)
            y_min = y_center - (height / 2)
            y_max = y_center + (height / 2)
            # Draw rectangle
            if class_label == 0:
                color = 'green'
            elif class_label == 1:
                color = 'red'
            else:
                color = 'yellow'
            draw.rectangle([x_min, y_min, x_max, y_max], outline=color, width=2)
    return image

In [ ]:
def show_sample_images_with_bounding_boxes(images_dir, labels_dir, num_samples=3):
    images, annotations = load_data(images_dir, labels_dir)
    
    sample_pairs = list(zip(images, annotations))[:num_samples]
    
    if not sample_pairs:
        print("No images found.")
        return

    # Display sample images
    plt.figure(figsize=(15, 6))
    for i, (image_path, annotation_path) in enumerate(sample_pairs):
        original_image = Image.open(image_path)
        annotated_image = draw_bounding_boxes(image_path, annotation_path)
        
        plt.subplot(2, num_samples, i + 1)
        plt.imshow(original_image)
        plt.axis('off')
        plt.title(f'Original {i+1}')
        
        # Plot Annotated
        plt.subplot(2, num_samples, num_samples + i + 1)
        plt.imshow(annotated_image)
        plt.axis('off')
        plt.title(f'Annotated {i+1}')
        
    plt.tight_layout()
    plt.show()

In [ ]:
show_sample_images_with_bounding_boxes(images_dir, labels_dir, num_samples=3)

If we look at the image distribution, we can see a class imbalance. To fix the class imbalance issue we will be using startification, which will distribute images equally. 

In [ ]:
from collections import Counter

base_working_dir = "/kaggle/working"

dirs = [
    os.path.join(base_working_dir, "train/images"),
    os.path.join(base_working_dir, "train/labels"),
    os.path.join(base_working_dir, "test/images"),
    os.path.join(base_working_dir, "test/labels")
]
for d in dirs: os.makedirs(d, exist_ok=True)



With class imbalance, we first need to map the number of images so that the split can happen equally.

In [ ]:
images = [os.path.join(images_dir, f) for f in os.listdir(images_dir) if f.endswith('.jpg')]

def get_image_class_mapping(images, labels_dir):
    mapping = {}
    for img_path in images:
        label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
        if os.path.exists(label_path):
            with open(label_path, 'r', encoding='utf-8-sig') as f:
                line = f.readline().strip()
                if line:
                    try:
                        mapping[img_path] = int(line.split()[0])
                    except (ValueError, IndexError):
                        mapping[img_path] = -1
                else:
                    mapping[img_path] = -1
        else:
            mapping[img_path] = -1
    return mapping

image_to_class = get_image_class_mapping(images, labels_dir)
valid_images = [img for img in images if image_to_class.get(img, -1) != -1]
stratify_labels = [image_to_class[img] for img in valid_images]

Let's split the image and copy the images and labels in train and test folder.

In [ ]:

train_images, test_images = train_test_split(
    valid_images, test_size=0.2, random_state=42, stratify=stratify_labels
)

def perform_split_copy(file_list, target_images_dir, target_labels_dir):
    for img_path in file_list:
        shutil.copy(img_path, os.path.join(target_images_dir, os.path.basename(img_path)))
        
        lbl_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path, os.path.join(target_labels_dir, os.path.basename(lbl_path)))

print("Copying files...")
perform_split_copy(train_images, dirs[0], dirs[1])
perform_split_copy(test_images, dirs[2], dirs[3])

In [ ]:
print(f"Training set: {len(train_images)} images.")
print(f"Testing set: {len(test_images)} images.")
print("Distribution in Training set:", Counter([image_to_class[img] for img in train_images]))

## 2. Model Creation
For our project on detecting various mosquito type, let's install YOLO and verify its functionality.

In [ ]:
# setup
%pip install ultralytics
import ultralytics
ultralytics.checks()

Next, let's create a configuration file for YOLO. This file provides information about the dataset and the classes it contains.

In [ ]:
dataset_yaml = """
train: /kaggle/working/train/images
val: /kaggle/working/test/images

nc: 3  # number of classes
names: ['anopheles_gambiae', 'anopheles_stephensi', 'aedes_aegypti']  # class names
"""

with open('/kaggle/working/dataset.yaml', 'w') as file:
    file.write(dataset_yaml)

Let's check the available commands in YOLOv8.

In [ ]:
!yolo

## 3. Model Training
We will train our model using `20` epochs with a batch size of `16`, and we will set the image size to `640`.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

model = YOLO('yolov8m.pt')
data_yaml_path = "/kaggle/working/dataset.yaml"

results = model.train(
    data=data_yaml_path,
    epochs=20,
    imgsz=640,
    device=[0,1],
    batch=32,   
    patience=15,
    save_period=10
)

## 4. Model Evaluation
Let's plot the testing results and then perform the evaluation on the validation set.

In [ ]:
from PIL import Image
tpaths2=[]
for dirname, _, filenames in os.walk('/kaggle/working/runs/detect/train'):
    for filename in filenames:
        if filename[-4:]=='.png' or filename[-4:]=='.jpg':
            tpaths2+=[(os.path.join(dirname, filename))]
tpaths2=sorted(tpaths2)

print(tpaths2[0])

In [ ]:
for path in tpaths2:
    image = Image.open(path)
    image=np.array(image)
    plt.figure(figsize=(20,10))
    plt.imshow(image)
    plt.show()


In [ ]:
# evaluate model performance on the validation set
results = model.val() 

## 5. Testing

Let's test our model with test images.

In [ ]:
images_path = '/kaggle/working/test/images'

available_files = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg'))]
print(f"Images found in directory: {len(available_files)}")

num_images = min(2, len(available_files))

if num_images == 0:
    print("No images found! Check your path.")
else:
    images = random.sample(available_files, num_images)

    for image in images:
        img_path = os.path.join(images_path, image)
        results = model(img_path, conf=0.5, iou=0.6)
        r = results[0]
        im_array = r.plot()
    
        display(Image.fromarray(im_array[..., ::-1]))

## 6. Mobile or embedded devices.

We need to export the models as `.tflite` and `.mlmodel` files to ensure they are compatible with mobile and embedded devices. To achieve this, we will use the Ultralytics YOLO CLI for model export.



In [ ]:
# Android
!yolo export format=tflite model="/kaggle/working/runs/detect/train/weights/best.pt" imgsz=320 int8 # For detection

In [ ]:
# iOS
!yolo export format=mlmodel model="/kaggle/working/runs/detect/train/weights/best.pt" imgsz=320,192 half nms